# Run the pipeline: facts → adapters → accuracy / lens / status tables

This notebook only wires the shared `xling` package together. Two datasets are supported:

| `DATASET` | Facts | Training | Reference for the lens |
|---|---|---|---|
| `"fictional"` (default) | `data/fictional_facts.jsonl`: the team's 50 fictional facts (v6.2) | round 1 (original fact) → checkpoint `round1`, then round 2 (update) → `round2` | `round1`: D = log p(round-2 answer) − log p(round-1 answer) |
| `"toy"` | `data/toy_facts.jsonl`: 5 real-world counterfactuals from the demo | one adapter `toy` | the base model |

The adapters trained here are **stand-ins** (`xling.toy_finetune`). When the real fine-tuning is ready, set `TRAIN_STANDIN = False` and point `ADAPTERS` at the real checkpoints. Each needs a `meta.json` with `name`, `fact_ids`, `reference` and `target` (see `README.md`).

Outputs go to `OUT` as CSV tables. `report_figures.ipynb` makes every figure from them without a GPU.

**Runtime:** GPU (T4 is fine). Upload the whole `64610` folder to `MyDrive/64610` first.

In [ ]:
# The project folder (with xling/, data/, notebooks/) lives in Google Drive.
# Change PROJECT_DIR if you put it somewhere else, or replace this cell with a `git clone`.
from google.colab import drive
drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/64610"
%cd {PROJECT_DIR}
import sys
sys.path.insert(0, PROJECT_DIR)

In [ ]:
!pip -q install -U transformers peft accelerate
# Colab preinstalls an old torchao that makes recent peft refuse to load; we don't use it.
!pip -q uninstall -y torchao

In [ ]:
BASE          = "Qwen/Qwen3-1.7B-Base"   # "Qwen/Qwen3-0.6B-Base" for a quick run
DATASET       = "fictional"              # or "toy"
TRAIN_STANDIN = True                     # set False once the real adapters exist

# Stand-in training (fictional): each round runs until TARGET_ACC of the English questions give
# that round's answer, then EXTRA_EPOCHS more, or until MAX_EPOCHS. Every epoch it also logs
# accuracy in every language, so you can see whether e.g. Finnish improves with more training.
MAX_EPOCHS, LR, TARGET_ACC, EXTRA_EPOCHS = 100, 2e-4, 0.9, 0

if DATASET == "fictional":
    FACTS    = "data/fictional_facts.jsonl"
    ADAPTERS = ["adapters/fictional/round1", "adapters/fictional/round2"]
    OUT      = "results/fictional"
else:
    FACTS    = "data/toy_facts.jsonl"
    ADAPTERS = ["adapters/toy"]
    OUT      = "results/toy"

# If the dataset JSON changes, regenerate the facts file first:
# !python data/fix_fictional_dataset.py && python data/convert_fictional.py

## 1. Load and validate
`validate_facts` is the same check the data team can run on their side. An empty list means the file is usable. The table shows the logit lens on one prompt (deepest layer on top).

In [ ]:
import pandas as pd
from IPython.display import display
from xling.modeling import load_model
from xling.schema import load_facts, validate_facts
from xling.plots import lens_grid

model, tok = load_model(BASE)
facts = load_facts(FACTS)
print(f"{len(facts)} facts; problems:", validate_facts(facts, tok) or "none")
print(facts[0].prompts["zh"][0])
display(lens_grid(model, tok, facts[0].prompts["zh"][0], every=2, last_n=6))

## 2. Stand-in adapters
**Fictional:** one LoRA, trained on round 1's sentences until `TARGET_ACC` of the English questions give the round-1 answer (saved as `round1`), then on round 2's until they give the round-2 answer (saved as `round2`). Whole-sentence loss, since the model has to learn the entities. Only English decides when to stop; the other languages are logged every epoch to `adapters/fictional/history.csv` and plotted below, but never used for stopping, so they stay a clean test of transfer.

On a T4 with the 1.7B model, expect roughly 20–40 s per epoch including the per-language check, so a full 100-epoch round takes up to about an hour. The earlier run hit its old 30-epoch cap at 78% (round 1) and 70% (round 2) English accuracy.

**Toy:** the demo's single counterfactual adapter.

Skip this once real adapters exist.

In [ ]:
from xling.toy_finetune import train_rounds, train_toy_adapter

if TRAIN_STANDIN:
    if DATASET == "fictional":
        pm = train_rounds(model, tok, facts, "adapters/fictional", base=BASE, lr=LR,
                          max_epochs=MAX_EPOCHS, target_acc=TARGET_ACC, extra_epochs=EXTRA_EPOCHS)
    else:
        pm = train_toy_adapter(model, tok, [f for f in facts if f.split == "edit"], "adapters/toy", name="toy", base=BASE)
    model = pm.unload()      # back to the plain base model for evaluation

In [ ]:
# Accuracy per language during training (fictional only).
import os
import matplotlib.pyplot as plt
from xling.plots import plot_training_history
if DATASET == "fictional" and os.path.exists("adapters/fictional/history.csv"):
    history = pd.read_csv("adapters/fictional/history.csv")
    plot_training_history(history); plt.show()

## 3. Evaluate
Scores the base model on every fact, then each adapter on the facts in its `meta.json`. `status.csv` judges each adapter against its reference: `round1` vs `base` (did the original fact get learned and transfer?), `round2` vs `round1` (did the update?).

In [ ]:
from xling.evaluate import run, summarize

acc, lens, status = run(FACTS, BASE, ADAPTERS, OUT, model=model, tok=tok)
display(summarize(status))

In [ ]:
# Per prompt, for the last adapter: what happened, and where in depth the training acted
# (see README for the failure-mode reading of these columns).
EDIT = status.model.iloc[-1]
cols = ["fact_id", "lang", "status", "onset_layer", "peak_dD", "final_dD", "final_D"]
display(status[status.model == EDIT][cols].round(1))

## 4. Hand-off to activation patching
The patching cases are the `transfer_fail` rows of the final model: learned in English, wrong in the target language. Patching should score its runs with `xling.scoring.score_logprobs` so its effect is in the same units as `D` here.

In [ ]:
patch_cases = status[(status.model == EDIT) & (status.status == "transfer_fail")][["model", "fact_id", "lang", "prompt_idx"]]
patch_cases.to_csv(f"{OUT}/patch_cases.csv", index=False)
print(len(patch_cases), "cases ->", f"{OUT}/patch_cases.csv")